# Attention Summarization Draft

Notebook nay dung de phan tich nhanh dataset `Content` -> `Summary` va ghi lai baseline attention truoc khi refactor vao `src/`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
train_path = DATA_DIR / 'train-00000-of-00001.parquet'
valid_path = DATA_DIR / 'valid-00000-of-00001.parquet'
test_path = DATA_DIR / 'test-00000-of-00001.parquet'

sample = pd.read_parquet(train_path, columns=['Content', 'Summary']).head(5)
sample

In [ ]:
def describe_lengths(path: Path, limit: int = 1000):
    frame = pd.read_parquet(path, columns=['Content', 'Summary']).head(limit)
    return pd.DataFrame({
        'content_tokens': frame['Content'].astype(str).str.split().str.len(),
        'summary_tokens': frame['Summary'].astype(str).str.split().str.len(),
    }).describe()

describe_lengths(train_path)

## Baseline idea

Pipeline trong `src/` dung encoder-decoder LSTM voi additive attention:

```text
Content tokens -> BiLSTM encoder -> attention context
Summary tokens -> LSTM decoder -> next summary token
```

Model nay la baseline de minh hoa attention mechanism truoc khi nang cap sang Transformer/BART/T5.

In [ ]:
# Quick smoke test from week4/Attention:
# !python src/main.py --epochs 1 --train-limit 200 --valid-limit 50 --test-limit 20 --batch-size 8 --device cpu

## Training process and results

Sau khi chay training, pipeline luu `output/history.csv`, `output/evaluation_report.json` va `output/generated_summaries.md`.

In [ ]:
history_path = PROJECT_ROOT / 'output' / 'history.csv'
if history_path.exists():
    history = pd.read_csv(history_path)
    display(history)
    history[['train_loss', 'valid_loss']].plot(figsize=(8, 4), title='Training history')
else:
    print('Run python src/main.py first to generate output/history.csv')

In [ ]:
report_path = PROJECT_ROOT / 'output' / 'evaluation_report.json'
samples_path = PROJECT_ROOT / 'output' / 'generated_summaries.md'
if report_path.exists():
    print(report_path.read_text(encoding='utf-8'))
if samples_path.exists():
    print(samples_path.read_text(encoding='utf-8')[:3000])